In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

RAW_PATH = "/Volumes/workspace/ingestion/project_data/raw"

print("Spark version:", spark.version)
print("Raw path:", RAW_PATH)

In [0]:
TOTAL_RECORDS = 10_000

base_df = (
    spark.range(1, TOTAL_RECORDS + 1)
    .withColumnRenamed("id", "row_id")
)

display(base_df.limit(10))

In [0]:
products = [
    ("P001", "iPhone 15", "Electronics"),
    ("P002", "Galaxy S24", "Electronics"),
    ("P003", "MacBook Air", "Electronics"),
    ("P004", "Running Shoes", "Sports"),
    ("P005", "Yoga Mat", "Sports"),
    ("P006", "Coffee Maker", "Home"),
    ("P007", "Office Chair", "Furniture"),
    ("P008", "Backpack", "Accessories"),
    ("P009", "T-Shirt", "Fashion"),
    ("P010", "Jeans", "Fashion"),
]

countries = [
    "India",
    "United States",
    "United Kingdom",
    "Singapore",
    "Australia",
    "Canada",
]

payment_methods = [
    "Credit Card",
    "Debit Card",
    "UPI",
    "Net Banking",
    "Cash on Delivery",
]

valid_statuses = [
    "Completed",
    "Cancelled",
    "Pending",
    "Returned",
]

product_ids = F.array(
    *[F.lit(x[0]) for x in products]
)

product_names = F.array(
    *[F.lit(x[1]) for x in products]
)

categories = F.array(
    *[F.lit(x[2]) for x in products]
)

country_array = F.array(
    *[F.lit(x) for x in countries]
)

payment_array = F.array(
    *[F.lit(x) for x in payment_methods]
)

status_array = F.array(
    *[F.lit(x) for x in valid_statuses]
)

In [0]:
orders_df = (
    base_df

    # --------------------------------------------------
    # Business identifiers
    # --------------------------------------------------

    .withColumn(
        "order_id",
        F.concat(
            F.lit("ORD"),
            F.lpad(
                F.col("row_id").cast("string"),
                7,
                "0"
            )
        )
    )

    .withColumn(
        "customer_id",
        F.concat(
            F.lit("CUST"),
            F.lpad(
                ((F.col("row_id") % 2500) + 1)
                .cast("string"),
                5,
                "0"
            )
        )
    )

    # --------------------------------------------------
    # Product
    # --------------------------------------------------

    .withColumn(
        "product_index",
        (
            (F.col("row_id") % len(products)) + 1
        ).cast("int")
    )

    .withColumn(
        "product_id",
        F.element_at(
            product_ids,
            F.col("product_index")
        )
    )

    .withColumn(
        "product_name",
        F.element_at(
            product_names,
            F.col("product_index")
        )
    )

    .withColumn(
        "category",
        F.element_at(
            categories,
            F.col("product_index")
        )
    )

    # --------------------------------------------------
    # Quantity
    # --------------------------------------------------

    .withColumn(
        "quantity",
        (
            (F.col("row_id") * 7) % 5 + 1
        ).cast("int")
    )

    # --------------------------------------------------
    # Price
    # --------------------------------------------------

    .withColumn(
        "unit_price",
        F.round(
            F.lit(20.0)
            + ((F.col("row_id") * 13) % 980),
            2
        )
    )

    # --------------------------------------------------
    # Discount
    # --------------------------------------------------

    .withColumn(
        "discount",
        F.round(
            (F.col("row_id") * 3) % 31,
            2
        )
    )

    # --------------------------------------------------
    # Order date
    # --------------------------------------------------

    .withColumn(
        "order_date",
        F.date_format(
            F.date_add(
                F.to_date(
                    F.lit("2026-01-01")
                ),
                (
                    F.col("row_id") % 181
                ).cast("int")
            ),
            "yyyy-MM-dd"
        )
    )

    # --------------------------------------------------
    # Country
    # --------------------------------------------------

    .withColumn(
        "country_index",
        (
            (F.col("row_id") % len(countries)) + 1
        ).cast("int")
    )

    .withColumn(
        "country",
        F.element_at(
            country_array,
            F.col("country_index")
        )
    )

    # --------------------------------------------------
    # Payment method
    # --------------------------------------------------

    .withColumn(
        "payment_index",
        (
            (F.col("row_id") % len(payment_methods)) + 1
        ).cast("int")
    )

    .withColumn(
        "payment_method",
        F.element_at(
            payment_array,
            F.col("payment_index")
        )
    )

    # --------------------------------------------------
    # Order status
    # --------------------------------------------------

    .withColumn(
        "status_index",
        (
            (F.col("row_id") % len(valid_statuses)) + 1
        ).cast("int")
    )

    .withColumn(
        "order_status",
        F.element_at(
            status_array,
            F.col("status_index")
        )
    )

    # --------------------------------------------------
    # Remove helper columns
    # --------------------------------------------------

    .drop(
        "product_index",
        "country_index",
        "payment_index",
        "status_index"
    )
)

display(orders_df.limit(10))

In [0]:
orders_df.printSchema()

In [0]:
orders_bad_df = (
    orders_df

    # Null customer IDs
    .withColumn(
        "customer_id",
        F.when(
            F.col("row_id") % 97 == 0,
            F.lit(None)
        ).otherwise(F.col("customer_id"))
    )

    # Missing product names
    .withColumn(
        "product_name",
        F.when(
            F.col("row_id") % 113 == 0,
            F.lit(None)
        ).otherwise(F.col("product_name"))
    )

    # Negative quantities
    .withColumn(
        "quantity",
        F.when(
            F.col("row_id") % 101 == 0,
            -F.abs(F.col("quantity"))
        ).otherwise(F.col("quantity"))
    )

    # Negative prices
    .withColumn(
        "unit_price",
        F.when(
            F.col("row_id") % 107 == 0,
            -F.abs(F.col("unit_price"))
        ).otherwise(F.col("unit_price"))
    )

    # Invalid discounts
    .withColumn(
        "discount",
        F.when(
            F.col("row_id") % 109 == 0,
            F.lit(110.0)
        )
        .when(
            F.col("row_id") % 127 == 0,
            F.lit(-5.0)
        )
        .otherwise(F.col("discount"))
    )

    # Invalid status
    .withColumn(
        "order_status",
        F.when(
            F.col("row_id") % 131 == 0,
            F.lit("Shipped")
        ).otherwise(F.col("order_status"))
    )

    # Invalid dates
    .withColumn(
        "order_date",
        F.when(
            F.col("row_id") % 137 == 0,
            F.lit("2026-02-30")
        )
        .when(
            F.col("row_id") % 149 == 0,
            F.lit("NOT_A_DATE")
        )
        .otherwise(F.col("order_date"))
    )
)

In [0]:
duplicate_rows = (
    orders_bad_df
    .filter(F.col("row_id") % 157 == 0)
    .limit(50)
)

duplicate_id_rows = (
    orders_bad_df
    .filter(F.col("row_id") % 163 == 0)
    .limit(50)
    .withColumn(
        "order_id",
        F.lit("ORD_DUPLICATE")
    )
)

final_orders_df = (
    orders_bad_df
    .unionByName(duplicate_rows)
    .unionByName(duplicate_id_rows)
)

print("Original records:", orders_df.count())
print("Final raw records:", final_orders_df.count())

In [0]:
business_columns = [
    "order_id",
    "customer_id",
    "order_date",
    "product_id",
    "product_name",
    "category",
    "quantity",
    "unit_price",
    "discount",
    "country",
    "payment_method",
    "order_status",
]

raw_orders_df = final_orders_df.select(business_columns)

display(raw_orders_df.limit(10))

Validate that bad data actually exists

In [0]:
print(
    "Null customer IDs:",
    raw_orders_df
    .filter(F.col("customer_id").isNull())
    .count()
)

In [0]:
print(
    "Missing product names:",
    raw_orders_df
    .filter(F.col("product_name").isNull())
    .count()
)

In [0]:
batch_1 = final_orders_df.filter(
    F.col("row_id") % 3 == 1
).select(business_columns)

batch_2 = final_orders_df.filter(
    F.col("row_id") % 3 == 2
).select(business_columns)

batch_3 = final_orders_df.filter(
    F.col("row_id") % 3 == 0
).select(business_columns)

In [0]:
print("Batch 1:", batch_1.count())
print("Batch 2:", batch_2.count())
print("Batch 3:", batch_3.count())

print(
    "Total:",
    batch_1.count()
    + batch_2.count()
    + batch_3.count()
)

In [0]:
RAW_PATH = "/Volumes/workspace/ingestion/project_data/raw"

In [0]:
(
    batch_1
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"{RAW_PATH}/orders_001")
)

(
    batch_2
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"{RAW_PATH}/orders_002")
)

(
    batch_3
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"{RAW_PATH}/orders_003")
)

print("All three raw batches written successfully.")

In [0]:
display(
    dbutils.fs.ls(RAW_PATH)
)

In [0]:
test_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{RAW_PATH}/orders_001")
)

display(test_raw_df.limit(10))
test_raw_df.printSchema()